# Autonomous Earnings Call Script & Presentation (IR Demo)

Multi-agent workflow for **Investor Relations** using:

- **Data source:** [Rogersurf/earnings-call-transcripts](https://huggingface.co/datasets/Rogersurf/earnings-call-transcripts) (research/education)
- **LLM:** Qwen3 via vLLM on AMD GPU
- **Agents:** Extract -> Predict hard questions -> Draft script, deck bullets, Q&A cheat sheet

> **Disclaimer:** Demo only — not for investor distribution.

## Step 1: Start vLLM (separate terminal)

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-30B-A3B \
    --served-model-name Qwen3-30B-A3B \
    --api-key abc-123 \
    --port 8000 \
    --enable-auto-tool-choice \
    --tool-call-parser hermes \
    --trust-remote-code
```

In [1]:
import os
import sys
from pathlib import Path

def resolve_agent_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if candidate.name == "notebook":
            parent = candidate.parent
            if (parent / "earnings_ir" / "pipeline.py").is_file():
                return parent
        if (candidate / "earnings_ir" / "pipeline.py").is_file() and candidate.name == "earnings-ir-agent":
            return candidate
        if (candidate / "agents" / "earnings-ir-agent" / "earnings_ir" / "pipeline.py").is_file():
            return candidate / "agents" / "earnings-ir-agent"
    raise RuntimeError(
        "Could not find earnings-ir-agent. Start Jupyter from the agent or TheRock repo."
    )


def ensure_agent_path() -> Path:
    global AGENT_ROOT, THEROCK_ROOT
    if "AGENT_ROOT" not in globals():
        AGENT_ROOT = resolve_agent_root()
        THEROCK_ROOT = (
            AGENT_ROOT.parent.parent
            if AGENT_ROOT.parent.name == "agents"
            else AGENT_ROOT.parent
        )
    agent_str = str(AGENT_ROOT)
    if agent_str not in sys.path:
        sys.path.insert(0, agent_str)
    return AGENT_ROOT


AGENT_ROOT = resolve_agent_root()
THEROCK_ROOT = AGENT_ROOT.parent.parent if AGENT_ROOT.parent.name == "agents" else AGENT_ROOT.parent
ensure_agent_path()

from earnings_ir.env_loader import load_agent_env

load_agent_env()

BASE_URL = os.environ.get("BASE_URL", "http://localhost:8000/v1")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "abc-123")
LLM_MODEL = os.environ.get("LLM_MODEL", "Qwen3-30B-A3B")

os.environ["BASE_URL"] = BASE_URL
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["LLM_MODEL"] = LLM_MODEL
os.environ["USE_LLM"] = os.environ.get("USE_LLM", "true")

TICKER = os.environ.get("DEFAULT_TICKER", "AMD")
print("AGENT_ROOT:", AGENT_ROOT)
print("THEROCK_ROOT:", THEROCK_ROOT)
print("Ticker:", TICKER)

Project root: C:\Users\Rajeswari\.gemini\antigravity\scratch\AMD-TCS-Hackthon\earnings-ir
Ticker: AMD


In [2]:
import httpx

response = httpx.get(
    f"{os.environ['BASE_URL']}/models",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    timeout=30.0,
)
response.raise_for_status()
print("Models:", [m["id"] for m in response.json().get("data", [])])

ConnectError: [WinError 10061] No connection could be made because the target machine actively refused it

## Step 2: Install dependencies

In [ ]:
!pip install -q -r "{AGENT_ROOT / 'requirements-notebook.txt'}"

## Step 3: Load earnings transcripts (HF or fallback cache)

In [ ]:
ensure_agent_path()
from earnings_ir.dataset import load_transcripts

records = load_transcripts(TICKER, limit=4)
print(f"Loaded {len(records)} transcript(s)")
for r in records:
    print(f"  - {r.quarter} {r.earnings_year}: {r.title[:70]}...")

## Step 4: Data Extraction Agent (Pydantic AI + tools)

In [ ]:
ensure_agent_path()
from earnings_ir.agents import build_agent_model, build_extraction_agent

model = build_agent_model()
extractor = build_extraction_agent(model)

In [ ]:
extraction = await extractor.run(
    f"List transcripts and show demo financials for {TICKER}"
)
print(extraction.output)

## Step 5: Full IR pipeline (Predictive Analyst + Drafting)

In [ ]:
ensure_agent_path()
from earnings_ir.pipeline import run_earnings_ir_pipeline

result = await run_earnings_ir_pipeline(TICKER)
print("LLM used:", result.llm_used)
print("Data source:", result.data_source)
print("Snippets used:", result.transcript_snippets_used)

In [ ]:
print("=== Predicted investor questions ===")
for i, q in enumerate(result.predicted_questions, 1):
    print(f"{i}. [{q.severity}] {q.question}")
    print(f"   Why: {q.rationale}\n")

In [ ]:
print("=== Earnings call script (opening) ===\n")
print(result.earnings_script)

print("\n=== Investor presentation bullets ===")
for b in result.presentation_bullets:
    print(" •", b)

print("\n=== Q&A cheat sheet (CEO/CFO) ===")
for item in result.qa_cheat_sheet[:5]:
    print(f"\nQ: {item.question}")
    print(f"A: {item.suggested_answer}")
    if item.talking_points:
        print("  Talking points:", "; ".join(item.talking_points))

## Step 6: Orchestrator agent (optional one-shot)

Ask the orchestrator agent to run the full pipeline via a single tool call.

In [ ]:
ensure_agent_path()
from earnings_ir.agents import build_orchestrator_agent

orchestrator = build_orchestrator_agent(model)
orch = await orchestrator.run(f"Prepare earnings IR materials for {TICKER}")
print(orch.output)